In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:18<00:00, 863.10it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:13<00:00, 12692.90it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:09<00:00, 18352.54it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.50, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'corona virus hospital rationing'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 41218/41218 [00:00<00:00, 71925.91it/s]


[('rfxzml1t', 13.313956590404189),
 ('w6ei0g42', 11.158569769696435),
 ('cmeuusz0', 11.139682867856191),
 ('2hcd5w9w', 10.741336232475783),
 ('a5aab4xx', 10.741336232475783),
 ('pjeef9s5', 10.683433827347514),
 ('f2zx6vuk', 10.683433827347514),
 ('y42zhjrt', 10.66936572584994),
 ('t1e262ar', 10.642908272854964),
 ('95hvdq23', 10.53275246189067),
 ('j8e8gb0l', 10.519963932688848),
 ('5nlk7a64', 10.491657190240215),
 ('pm713jib', 10.426600009043252),
 ('onr700ue', 10.38618806997647),
 ('03eod3df', 10.363411275879841),
 ('vjtnjmni', 10.268581760658481),
 ('uvac32oo', 10.2488098993859),
 ('t55od92g', 10.17193852948555),
 ('vsinwqnr', 10.143836773366484),
 ('od8k0utb', 9.946644999417545),
 ('8dico3zc', 9.946644999417545),
 ('gy8f1oyt', 9.89627098228872),
 ('bac8bbql', 9.89627098228872),
 ('0bb729sp', 9.868869917868151),
 ('e5txqhml', 9.855862252495578),
 ('0vueeyub', 9.8497779916629),
 ('5nnax458', 9.836828220267307),
 ('bqhmxzrj', 9.83294486188625),
 ('kjeqs6zh', 9.816797379261471),
 ('sp7